# Agente de Riesgo Crediticio (Chat con Gradio + LangChain + Qwen)

In [40]:
import json
import os
import re
from datetime import datetime, timezone

import gradio as gr
import requests
from sqlalchemy import Column, DateTime, Float, Integer, String, create_engine
from sqlalchemy.orm import declarative_base, sessionmaker

## Configuracion de la base de datos (SQLite + SQLAlchemy)

In [41]:
DATABASE_URL = "sqlite:///./credit_risk.db"

engine = create_engine(DATABASE_URL, connect_args={"check_same_thread": False})
SessionLocal = sessionmaker(autocommit=False, autoflush=False, bind=engine)
Base = declarative_base()


class Evaluation(Base):
    __tablename__ = "evaluations"

    id = Column(Integer, primary_key=True, index=True)
    interest_rate = Column(Float, nullable=False)
    num_credit_inquiries = Column(Float, nullable=False)
    outstanding_debt = Column(Float, nullable=False)
    delay_from_due_date = Column(Float, nullable=False)
    num_credit_card = Column(Float, nullable=False)
    predicted_category = Column(Integer, nullable=False)
    decision = Column(String, nullable=False)
    created_at = Column(DateTime, default=lambda: datetime.now(timezone.utc))


Base.metadata.create_all(bind=engine)

## Configuracion del LLM (Qwen, gratuito y local via Ollama)

In [42]:
API_URL = os.environ.get("CREDIT_API_URL", "http://127.0.0.1:8000/predict")
QWEN_MODEL = os.environ.get("QWEN_MODEL", "qwen2.5:3b")

try:
    from langchain_ollama import ChatOllama
    llm = ChatOllama(model=QWEN_MODEL, temperature=0.2)
except Exception:
    llm = None

## Reglas de decision y funciones del agente

In [43]:
RISK_LEVEL_NAMES = {0: "bajo", 1: "medio", 2: "alto"}

DECISION_RULES = {
    0: "Aprobar solicitud con condiciones estandar.",
    1: "Solicitar documentacion adicional y evaluar nuevamente.",
    2: "Rechazar solicitud y recomendar educacion financiera.",
}

#Las mismas 5 caracteristicas con mas peso seleccionadas en train_model.ipynb
CAMPOS_REQUERIDOS = {
    "interest_rate": "tasa de interes",
    "num_credit_inquiries": "numero de consultas de credito",
    "outstanding_debt": "deuda pendiente",
    "delay_from_due_date": "dias de atraso en pagos",
    "num_credit_card": "numero de tarjetas de credito",
}

#Qwen (3B) no siempre entiende que una negacion equivale a 0 (ej. "sin atrasos").
#Estos patrones son un respaldo por regex para esos casos comunes.
CAMPOS_NEGACION = {
    "delay_from_due_date": r"sin\s+atras|no\s+(me\s+)?he?\s+atrasad|nunca\s+.*atras|no\s+tengo\s+atras",
    "num_credit_inquiries": r"sin\s+consultas?|ninguna\s+consulta|no\s+he\s+tenido\s+consultas?",
    "outstanding_debt": r"sin\s+deuda|no\s+debo|no\s+tengo\s+deuda",
    "num_credit_card": r"sin\s+tarjetas?|no\s+tengo\s+tarjetas?|ninguna\s+tarjeta",
}

## Extraccion de datos desde el chat

In [44]:
def extraer_datos_solicitante(texto_conversacion):
    if llm is None:
        return None

    prompt = (
        "Extrae de la siguiente conversacion los datos financieros del "
        "solicitante de credito. Identifica, si aparecen, estos 5 valores "
        "numericos:\n"
        "- interest_rate: tasa de interes\n"
        "- num_credit_inquiries: numero de consultas de credito\n"
        "- outstanding_debt: deuda pendiente\n"
        "- delay_from_due_date: dias de atraso en pagos\n"
        "- num_credit_card: numero de tarjetas de credito\n\n"
        "Responde UNICAMENTE con un JSON valido, sin texto adicional ni bloques "
        "de markdown, con esta forma exacta (usa null si un dato no aparece):\n"
        '{"interest_rate": null, "num_credit_inquiries": null, '
        '"outstanding_debt": null, "delay_from_due_date": null, '
        '"num_credit_card": null}'
        "\n\nConversacion del solicitante:\n" + texto_conversacion
    )

    respuesta = llm.invoke(prompt)
    texto = (respuesta.content or "").strip()

    coincidencia = re.search(r"\{.*\}", texto, re.DOTALL)
    if not coincidencia:
        return None

    try:
        datos = json.loads(coincidencia.group(0))
    except json.JSONDecodeError:
        return None

    #Si Qwen devuelve algo no numerico (ej. "20%"), se trata como dato faltante
    #en vez de mandarlo asi a la API.
    for campo in CAMPOS_REQUERIDOS:
        valor = datos.get(campo)
        if valor is None:
            continue
        try:
            datos[campo] = float(re.sub(r"[^0-9.\-]", "", str(valor)))
        except ValueError:
            datos[campo] = None

    #Respaldo: si el campo sigue sin dato pero el texto trae una negacion
    #reconocida (ej. "sin atrasos"), se asume 0 en vez de pedirselo de nuevo.
    for campo, patron in CAMPOS_NEGACION.items():
        if datos.get(campo) is None and re.search(patron, texto_conversacion, re.IGNORECASE):
            datos[campo] = 0.0

    return datos

In [45]:
def campos_faltantes(datos):
    return [nombre for campo, nombre in CAMPOS_REQUERIDOS.items() if datos.get(campo) is None]

In [46]:
def call_prediction_api(interest_rate, num_credit_inquiries, outstanding_debt,
                         delay_from_due_date, num_credit_card):
    payload = {
        "interest_rate": interest_rate,
        "num_credit_inquiries": num_credit_inquiries,
        "outstanding_debt": outstanding_debt,
        "delay_from_due_date": delay_from_due_date,
        "num_credit_card": num_credit_card,
    }
    response = requests.post(API_URL, json=payload, timeout=10)
    response.raise_for_status()
    return response.json()

In [47]:
def generate_justification(category, decision):
    fallback = (
        f"El sistema clasifico la solicitud en riesgo {RISK_LEVEL_NAMES[category]} "
        f"({category}). Segun la regla de negocio establecida para esta categoria, "
        f"la decision tomada es: {decision}"
    )

    if llm is None:
        return fallback

    prompt = (
        "Eres un asistente de un banco. Debes justificar en 2 o 3 oraciones una "
        "decision de credito. Basate UNICAMENTE en la categoria de riesgo obtenida "
        "y en la regla de decision aplicada, que se listan abajo. No inventes ni "
        "menciones valores numericos de variables del solicitante, no expliques el "
        "porque matematico o estadistico de la categoria.\n\n"
        f"Categoria de riesgo: {category} ({RISK_LEVEL_NAMES[category]})\n"
        f"Regla de decision aplicada: {decision}\n\n"
        "Escribe la justificacion en espa\u00f1ol, en tono profesional."
    )

    try:
        result = llm.invoke(prompt)
        text = getattr(result, "content", None)
        return text.strip() if text else fallback
    except Exception as exc:
        return f"{fallback}\n\n(Nota: el LLM Qwen no estuvo disponible: {exc})"

In [48]:
def log_evaluation(interest_rate, num_credit_inquiries, outstanding_debt,
                    delay_from_due_date, num_credit_card, category, decision):
    session = SessionLocal()
    try:
        record = Evaluation(
            interest_rate=interest_rate,
            num_credit_inquiries=num_credit_inquiries,
            outstanding_debt=outstanding_debt,
            delay_from_due_date=delay_from_due_date,
            num_credit_card=num_credit_card,
            predicted_category=category,
            decision=decision,
        )
        session.add(record)
        session.commit()
    finally:
        session.close()

## Funcion principal del chat

In [49]:
def extraer_texto(contenido):
    #Gradio a veces entrega el contenido del mensaje como texto plano y a veces
    #como una lista de bloques (ej. adjuntos); aqui se normaliza a texto plano.
    if isinstance(contenido, str):
        return contenido
    if isinstance(contenido, list):
        partes = []
        for item in contenido:
            if isinstance(item, str):
                partes.append(item)
            elif isinstance(item, dict) and "text" in item:
                partes.append(str(item["text"]))
        return " ".join(partes)
    return str(contenido)

In [50]:
def mensajes_desde_ultima_evaluacion(history):
    #Si ya se completo una evaluacion antes en esta misma conversacion, sus datos
    #no deben mezclarse con una solicitud nueva: se reinicia el contexto despues
    #de cada respuesta que empiece con 'Categoria de riesgo:'.
    mensajes = []
    for turno in history:
        contenido = extraer_texto(turno.get("content"))
        if turno.get("role") == "assistant" and contenido.startswith("Categoria de riesgo:"):
            mensajes = []
        elif turno.get("role") == "user":
            mensajes.append(contenido)
    return mensajes

In [51]:
def responder(message, history):
    if llm is None:
        return (
            "Este chat necesita el modelo Qwen (via Ollama) para entender tu "
            "mensaje. Verifica que Ollama este corriendo y que hayas descargado "
            "el modelo con 'ollama pull qwen2.5:3b'."
        )

    mensajes_usuario = mensajes_desde_ultima_evaluacion(history)
    mensajes_usuario.append(extraer_texto(message))
    conversacion = "\n".join(mensajes_usuario)

    datos = extraer_datos_solicitante(conversacion)
    if datos is None:
        return "No pude entender los datos de tu mensaje. ¿Puedes intentar describirlos de nuevo?"

    faltantes = campos_faltantes(datos)
    if faltantes:
        return "Para evaluar tu solicitud todavia necesito que me digas: " + ", ".join(faltantes) + "."

    try:
        prediction = call_prediction_api(
            datos["interest_rate"], datos["num_credit_inquiries"], datos["outstanding_debt"],
            datos["delay_from_due_date"], datos["num_credit_card"],
        )
    except requests.exceptions.HTTPError as exc:
        try:
            detalle = exc.response.json().get("detail", exc.response.text)
        except Exception:
            detalle = str(exc)
        return (
            "Los datos que entendi no son validos para la API (" + str(detalle) + "). Puedes describirlos de nuevo, por ejemplo: "
            "'tasa de interes 15, 3 consultas de credito, debo 800, 10 dias de atraso, 4 tarjetas'."
        )
    except requests.exceptions.RequestException as exc:
        return f"Error al contactar la API de prediccion: {exc}"

    category = prediction["risk_category"]
    risk_level = prediction["risk_level"]
    decision = DECISION_RULES[category]
    justification = generate_justification(category, decision)

    log_evaluation(
        datos["interest_rate"], datos["num_credit_inquiries"], datos["outstanding_debt"],
        datos["delay_from_due_date"], datos["num_credit_card"], category, decision,
    )

    return (
        f"Categoria de riesgo: {category} ({risk_level})\n\n"
        f"Decision: {decision}\n\n"
        f"Justificacion:\n{justification}"
    )

## Interfaz de chat (Gradio)

In [52]:
demo = gr.ChatInterface(
    fn=responder,
    title="Evaluacion de Riesgo Crediticio",
    description=(
        "Cuentame tu situacion crediticia: tasa de interes, numero de consultas de "
        "credito, deuda pendiente, dias de atraso en pagos y numero de tarjetas de "
        "credito. Puedes escribirlo todo junto o poco a poco."
    ),
)

## Lanzar la interfaz

In [53]:
demo.launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.
